# ZenPose Template Training & Evaluation Notebook

This notebook runs your full training/evaluation flow and compares:
- **Euclidean nearest-template**
- **Cosine nearest-template**

Outputs:
1. `assets/pose_templates.json`
2. `build/pose_eval/split_manifest.json`
3. `build/pose_eval/evaluation_report.json`


## Algorithm Summary

### Feature extraction
- Detect 33 landmarks with MediaPipe Pose.
- Keep 12 key joints.
- Normalize by hip-center translation + torso-length scaling.
- Final feature vector: **24 values**.

### Training (prototype model)
- For each class, average all train vectors.
- Class mean vector = class template.

### Inference
- Compare sample vector to each template.
- Euclidean mode: pick smallest distance.
- Cosine mode: pick highest cosine similarity.


In [27]:
# Optional one-time install
# !pip install mediapipe opencv-python numpy


In [28]:
from __future__ import annotations

import json
import math
import random
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import cv2
import mediapipe as mp
import numpy as np


In [30]:
# --------------------------
# Configuration
# --------------------------
DATASET_DIR = Path("C:/Users/danis/Documents/FYP/zenpose/dataset")
OUTPUT_FILE = Path("assets/pose_templates.json")
SPLIT_MANIFEST_FILE = Path("build/pose_eval/split_manifest.json")
EVALUATION_REPORT_FILE = Path("build/pose_eval/evaluation_report.json")

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15
SEED = 42

MIN_DETECTION_CONFIDENCE = 0.5
MIN_LANDMARK_VISIBILITY = 0.5
MIN_TORSO_LENGTH = 1e-6
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

KEY_JOINT_INDICES = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
LEFT_SHOULDER_IDX = 11
RIGHT_SHOULDER_IDX = 12
LEFT_HIP_IDX = 23
RIGHT_HIP_IDX = 24

SPLITS = ("train", "validation", "test")


In [31]:
@dataclass(frozen=True)
class Sample:
    class_name: str
    relative_path: str
    split: str


@dataclass(frozen=True)
class ValidExample:
    class_name: str
    split: str
    relative_path: str
    vector: list[float]


def validate_split_ratios(train_ratio: float, validation_ratio: float, test_ratio: float) -> None:
    total = train_ratio + validation_ratio + test_ratio
    if not math.isclose(total, 1.0, rel_tol=1e-6, abs_tol=1e-6):
        raise ValueError(f"Split ratios must sum to 1.0, got {total:.6f}")


def collect_image_paths(class_dir: Path) -> list[Path]:
    return sorted(
        p for p in class_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )


def get_dataset_index(dataset_dir: Path) -> dict[str, list[Path]]:
    if not dataset_dir.is_dir():
        raise FileNotFoundError(f"Dataset directory not found: {dataset_dir}")

    class_dirs = sorted(d for d in dataset_dir.iterdir() if d.is_dir())
    class_to_images: dict[str, list[Path]] = {}
    for class_dir in class_dirs:
        images = collect_image_paths(class_dir)
        if images:
            class_to_images[class_dir.name] = images

    if not class_to_images:
        raise RuntimeError("No image files found in dataset.")
    return class_to_images


def split_counts_for_class(n: int, train_ratio: float, validation_ratio: float, test_ratio: float) -> tuple[int, int, int]:
    if n <= 0:
        return (0, 0, 0)
    if n == 1:
        return (1, 0, 0)
    if n == 2:
        return (1, 1, 0)

    raw = np.array([train_ratio, validation_ratio, test_ratio], dtype=float) * n
    counts = np.floor(raw).astype(int)
    remainder = n - int(counts.sum())

    if remainder > 0:
        for idx in np.argsort(-(raw - counts))[:remainder]:
            counts[idx] += 1

    for idx in (0, 1, 2):
        if counts[idx] == 0:
            donor = int(np.argmax(counts))
            if counts[donor] > 1:
                counts[donor] -= 1
                counts[idx] += 1

    train_count, val_count, test_count = map(int, counts)
    if train_count == 0 and n > 0:
        train_count = 1
        if val_count > test_count and val_count > 0:
            val_count -= 1
        elif test_count > 0:
            test_count -= 1

    if train_count + val_count + test_count != n:
        test_count = n - train_count - val_count

    return train_count, val_count, test_count


def build_split_manifest(
    class_to_images: dict[str, list[Path]],
    dataset_dir: Path,
    train_ratio: float,
    validation_ratio: float,
    test_ratio: float,
    seed: int,
) -> list[Sample]:
    rng = random.Random(seed)
    samples: list[Sample] = []

    for class_name, image_paths in sorted(class_to_images.items()):
        shuffled = image_paths[:]
        rng.shuffle(shuffled)

        train_count, val_count, test_count = split_counts_for_class(
            len(shuffled), train_ratio, validation_ratio, test_ratio
        )

        split_labels = (
            ["train"] * train_count
            + ["validation"] * val_count
            + ["test"] * test_count
        )

        for image_path, split in zip(shuffled, split_labels):
            rel = image_path.relative_to(dataset_dir).as_posix()
            samples.append(Sample(class_name=class_name, relative_path=rel, split=split))

    return samples


def save_split_manifest(path: Path, samples: list[Sample]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "createdAtUtc": datetime.now(timezone.utc).isoformat(),
        "seed": SEED,
        "ratios": {
            "train": TRAIN_RATIO,
            "validation": VALIDATION_RATIO,
            "test": TEST_RATIO,
        },
        "samples": [
            {"class": s.class_name, "path": s.relative_path, "split": s.split}
            for s in samples
        ],
    }
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


In [32]:
def normalize_landmarks(landmarks: Any) -> list[float] | None:
    left_hip = landmarks[LEFT_HIP_IDX]
    right_hip = landmarks[RIGHT_HIP_IDX]
    left_sh = landmarks[LEFT_SHOULDER_IDX]
    right_sh = landmarks[RIGHT_SHOULDER_IDX]

    for lm in (left_hip, right_hip, left_sh, right_sh):
        if lm.visibility < MIN_LANDMARK_VISIBILITY:
            return None

    hip_cx = (left_hip.x + right_hip.x) / 2.0
    hip_cy = (left_hip.y + right_hip.y) / 2.0
    sh_cx = (left_sh.x + right_sh.x) / 2.0
    sh_cy = (left_sh.y + right_sh.y) / 2.0

    dx = sh_cx - hip_cx
    dy = sh_cy - hip_cy
    torso_length = math.sqrt(dx * dx + dy * dy)
    if torso_length < MIN_TORSO_LENGTH:
        return None

    vector: list[float] = []
    for idx in KEY_JOINT_INDICES:
        lm = landmarks[idx]
        if lm.visibility < MIN_LANDMARK_VISIBILITY:
            return None
        vector.append((lm.x - hip_cx) / torso_length)
        vector.append((lm.y - hip_cy) / torso_length)

    return vector


def extract_valid_examples(dataset_dir: Path, samples: list[Sample]) -> tuple[list[ValidExample], dict[str, Any]]:
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(
        static_image_mode=True,
        model_complexity=2,
        min_detection_confidence=MIN_DETECTION_CONFIDENCE,
    )

    valid_examples: list[ValidExample] = []
    summary: dict[str, Any] = {
        "bySplit": {split: {"total": 0, "valid": 0, "invalid": 0} for split in SPLITS},
        "invalidReasonsBySplit": {split: defaultdict(int) for split in SPLITS},
    }

    for sample in samples:
        split = sample.split
        summary["bySplit"][split]["total"] += 1

        image_path = dataset_dir / sample.relative_path
        bgr = cv2.imread(str(image_path))
        if bgr is None:
            summary["bySplit"][split]["invalid"] += 1
            summary["invalidReasonsBySplit"][split]["unreadableImage"] += 1
            continue

        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        result = pose.process(rgb)

        if result.pose_landmarks is None:
            summary["bySplit"][split]["invalid"] += 1
            summary["invalidReasonsBySplit"][split]["poseNotDetected"] += 1
            continue

        vector = normalize_landmarks(result.pose_landmarks.landmark)
        if vector is None:
            summary["bySplit"][split]["invalid"] += 1
            summary["invalidReasonsBySplit"][split]["lowConfidenceOrDegenerate"] += 1
            continue

        summary["bySplit"][split]["valid"] += 1
        valid_examples.append(
            ValidExample(
                class_name=sample.class_name,
                split=sample.split,
                relative_path=sample.relative_path,
                vector=vector,
            )
        )

    pose.close()
    summary["invalidReasonsBySplit"] = {
        split: dict(reasons)
        for split, reasons in summary["invalidReasonsBySplit"].items()
    }
    return valid_examples, summary


In [33]:
def build_templates_from_train(valid_examples: list[ValidExample]) -> dict[str, dict[str, list[float]]]:
    vectors_by_class: dict[str, list[list[float]]] = defaultdict(list)
    for ex in valid_examples:
        if ex.split == "train":
            vectors_by_class[ex.class_name].append(ex.vector)

    templates: dict[str, dict[str, list[float]]] = {}
    for class_name, vectors in sorted(vectors_by_class.items()):
        if not vectors:
            continue
        matrix = np.array(vectors)
        mean_vector = matrix.mean(axis=0)
        templates[class_name] = {"meanVector": [round(float(v), 6) for v in mean_vector]}

    return templates


def euclidean_distance(vec_a: list[float], vec_b: list[float]) -> float:
    return float(np.linalg.norm(np.array(vec_a, dtype=float) - np.array(vec_b, dtype=float)))


def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    a = np.array(vec_a, dtype=float)
    b = np.array(vec_b, dtype=float)
    denom = float(np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0.0:
        return 0.0
    score = float(np.dot(a, b) / denom)
    return max(-1.0, min(1.0, score))


def predict_class(vector: list[float], templates: dict[str, dict[str, list[float]]], metric: str = "euclidean") -> str:
    best_label = ""

    if metric == "euclidean":
        best_value = float("inf")
        for class_name, payload in templates.items():
            dist = euclidean_distance(vector, payload["meanVector"])
            if dist < best_value:
                best_value = dist
                best_label = class_name
    elif metric == "cosine":
        best_value = -float("inf")
        for class_name, payload in templates.items():
            sim = cosine_similarity(vector, payload["meanVector"])
            if sim > best_value:
                best_value = sim
                best_label = class_name
    else:
        raise ValueError("metric must be 'euclidean' or 'cosine'")

    return best_label


def safe_divide(a: float, b: float) -> float:
    return 0.0 if b == 0 else a / b


def evaluate_split(split_name: str, valid_examples: list[ValidExample], templates: dict[str, dict[str, list[float]]], metric: str) -> dict[str, Any]:
    labels = sorted(templates.keys())
    label_to_idx = {label: i for i, label in enumerate(labels)}

    split_examples = [ex for ex in valid_examples if ex.split == split_name]
    y_true: list[str] = []
    y_pred: list[str] = []

    for ex in split_examples:
        if ex.class_name not in label_to_idx:
            continue
        y_true.append(ex.class_name)
        y_pred.append(predict_class(ex.vector, templates, metric=metric))

    matrix = np.zeros((len(labels), len(labels)), dtype=int)
    for true_label, pred_label in zip(y_true, y_pred):
        matrix[label_to_idx[true_label], label_to_idx[pred_label]] += 1

    total = int(matrix.sum())
    correct = int(np.trace(matrix))
    accuracy = safe_divide(correct, total)

    per_class: dict[str, dict[str, Any]] = {}
    precision_values: list[float] = []
    recall_values: list[float] = []
    f1_values: list[float] = []

    for label in labels:
        idx = label_to_idx[label]
        tp = int(matrix[idx, idx])
        fp = int(matrix[:, idx].sum() - tp)
        fn = int(matrix[idx, :].sum() - tp)
        support = int(matrix[idx, :].sum())

        precision = safe_divide(tp, tp + fp)
        recall = safe_divide(tp, tp + fn)
        f1 = safe_divide(2 * precision * recall, precision + recall) if (precision + recall) > 0 else 0.0

        if support > 0:
            precision_values.append(precision)
            recall_values.append(recall)
            f1_values.append(f1)

        per_class[label] = {
            "precision": round(precision, 6),
            "recall": round(recall, 6),
            "f1": round(f1, 6),
            "support": support,
        }

    macro_precision = float(np.mean(precision_values)) if precision_values else 0.0
    macro_recall = float(np.mean(recall_values)) if recall_values else 0.0
    macro_f1 = float(np.mean(f1_values)) if f1_values else 0.0

    confusion_matrix = {
        "labels": labels,
        "rows": {
            true_label: {
                pred_label: int(matrix[label_to_idx[true_label], label_to_idx[pred_label]])
                for pred_label in labels
            }
            for true_label in labels
        },
    }

    return {
        "metric": metric,
        "split": split_name,
        "numValidExamples": len(split_examples),
        "numEvaluatedExamples": total,
        "accuracy": round(accuracy, 6),
        "precision": round(macro_precision, 6),
        "recall": round(macro_recall, 6),
        "f1": round(macro_f1, 6),
        "perClass": per_class,
        "confusionMatrix": confusion_matrix,
    }


In [34]:
# --------------------------
# Execute pipeline
# --------------------------
validate_split_ratios(TRAIN_RATIO, VALIDATION_RATIO, TEST_RATIO)
class_to_images = get_dataset_index(DATASET_DIR)
samples = build_split_manifest(
    class_to_images=class_to_images,
    dataset_dir=DATASET_DIR,
    train_ratio=TRAIN_RATIO,
    validation_ratio=VALIDATION_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED,
)
save_split_manifest(SPLIT_MANIFEST_FILE, samples)

print(f"Split manifest saved: {SPLIT_MANIFEST_FILE}")
print(f"Total samples indexed: {len(samples)}")


Split manifest saved: build\pose_eval\split_manifest.json
Total samples indexed: 1959


In [35]:
# Split counts
split_table: dict[str, dict[str, int]] = defaultdict(lambda: defaultdict(int))
for s in samples:
    split_table[s.class_name][s.split] += 1

print(f"{'Class':<18} {'Train':>8} {'Validation':>12} {'Test':>8} {'Total':>10}")
print("-" * 72)
for class_name in sorted(split_table.keys()):
    tr = split_table[class_name].get("train", 0)
    va = split_table[class_name].get("validation", 0)
    te = split_table[class_name].get("test", 0)
    print(f"{class_name:<18} {tr:>8} {va:>12} {te:>8} {tr+va+te:>10}")


Class                 Train   Validation     Test      Total
------------------------------------------------------------------------
chair                    57           12       12         81
downdog                 276           59       59        394
goddess                 182           39       39        260
half-moon                41            9        9         59
plank                   313           67       67        447
tree                    208           45       44        297
warrior2                295           63       63        421


In [36]:
valid_examples, extraction_summary = extract_valid_examples(DATASET_DIR, samples)
print("Extraction summary by split:")
for split in SPLITS:
    d = extraction_summary["bySplit"][split]
    print(f"  {split:<10} total={d['total']:<6} valid={d['valid']:<6} invalid={d['invalid']:<6}")


c:\Users\danis\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Extraction summary by split:
  train      total=1372   valid=640    invalid=732   
  validation total=294    valid=138    invalid=156   
  test       total=293    valid=132    invalid=161   


In [37]:
templates = build_templates_from_train(valid_examples)
if not templates:
    raise RuntimeError("No templates generated. Check training split / image quality.")

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    json.dump(templates, f, indent=2)

print(f"Templates saved: {OUTPUT_FILE}")
print(f"Classes trained: {len(templates)}")


Templates saved: assets\pose_templates.json
Classes trained: 7


In [38]:
# Evaluate both metrics on validation and test
results = {
    "euclidean": {
        "validation": evaluate_split("validation", valid_examples, templates, metric="euclidean"),
        "test": evaluate_split("test", valid_examples, templates, metric="euclidean"),
    },
    "cosine": {
        "validation": evaluate_split("validation", valid_examples, templates, metric="cosine"),
        "test": evaluate_split("test", valid_examples, templates, metric="cosine"),
    },
}

report = {
    "createdAtUtc": datetime.now(timezone.utc).isoformat(),
    "datasetDir": str(DATASET_DIR),
    "outputFile": str(OUTPUT_FILE),
    "splitManifestFile": str(SPLIT_MANIFEST_FILE),
    "ratios": {
        "train": TRAIN_RATIO,
        "validation": VALIDATION_RATIO,
        "test": TEST_RATIO,
    },
    "seed": SEED,
    "extractionSummary": extraction_summary,
    "numTemplates": len(templates),
    "evaluation": results,
}

EVALUATION_REPORT_FILE.parent.mkdir(parents=True, exist_ok=True)
with EVALUATION_REPORT_FILE.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(f"Evaluation report saved: {EVALUATION_REPORT_FILE}")


Evaluation report saved: build\pose_eval\evaluation_report.json


In [39]:
# Final metric values only
print(f"{'Metric':<12} {'Split':<12} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 72)
for metric_name in ("euclidean", "cosine"):
    for split_name in ("validation", "test"):
        m = results[metric_name][split_name]
        print(
            f"{metric_name:<12} {split_name:<12} "
            f"{m['accuracy']:>10.4f} {m['precision']:>10.4f} {m['recall']:>10.4f} {m['f1']:>10.4f}"
        )


Metric       Split          Accuracy  Precision     Recall         F1
------------------------------------------------------------------------
euclidean    validation       0.7101     0.6771     0.6114     0.6399
euclidean    test             0.6515     0.4580     0.4122     0.4291
cosine       validation       0.7319     0.7524     0.6329     0.6828
cosine       test             0.6591     0.4763     0.4241     0.4384


## Notes
- `accuracy/precision/recall/F1` shown above are **macro averages** across classes.
- Use per-class metrics inside `results[metric][split]['perClass']` for detailed error analysis.
